实现共享参数绑定，再验证参数身份、两条路径梯度相加及更新后共享。 本Notebook在Python环境运行，GitHub Pages不执行PyTorch或真实模型。先完成TODO，再运行验证；最后的参考实现可按需查看。


In [ ]:
%pip install torch


In [ ]:
"""PyTorch共享参数实验。安装: pip install torch
任务：补全build_tied，用同一Parameter绑定Embedding与LM Head，再验证梯度。
参考实现默认已给出，先注释赋值行自行完成。
"""
import torch
from torch import nn

def build_tied(vocab=5,dim=3):
    # TODO：完成本实验目标，再执行下方验证。
    raise NotImplementedError("请完成 build_tied")

def verify():
    torch.manual_seed(7)
    embedding,head=build_tied()
    assert embedding.weight is head.weight
    weight=embedding.weight.detach().clone()
    ids=torch.tensor([0,2,0])
    loss=head(embedding(ids)).square().sum()
    loss.backward()
    shared=embedding.weight.grad.clone()
    # 拆开为独立参数，得到输入路径与输出路径的两份梯度。
    input_weight=weight.clone().requires_grad_()
    output_weight=weight.clone().requires_grad_()
    independent=torch.nn.functional.embedding(ids,input_weight) @ output_weight.T
    independent.square().sum().backward()
    assert torch.allclose(shared,input_weight.grad+output_weight.grad,atol=1e-10)
    optimizer=torch.optim.SGD(embedding.parameters(),lr=.001)
    optimizer.step()
    assert embedding.weight is head.weight
    assert not torch.equal(embedding.weight.detach(),weight)
    return {'parameter_identity':True,'gradient_sum_error':float((shared-input_weight.grad-output_weight.grad).abs().max()),'shared_after_update':True}



完成目标函数后，运行以下验证。初始代码出现NotImplementedError是预期状态。


In [ ]:
verify()


参考实现：先自行完成，再运行下一格替换目标函数，对照结果。


In [ ]:
def build_tied(vocab=5,dim=3):
    embedding=nn.Embedding(vocab,dim,dtype=torch.float64)
    head=nn.Linear(dim,vocab,bias=False,dtype=torch.float64)
    head.weight=embedding.weight
    return embedding,head


In [ ]:
verify()
